In [9]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/deepSeekR1Data.xlsx" .

In [10]:
!pip install hazm bert_score

In [11]:
!pip install -q "transformers<5" "tokenizers<0.22" "bert-score==0.3.12"

In [12]:
import numpy as np
import pandas as pd
import ast
from hazm import WordTokenizer
from bert_score import score as bert_score
import torch

In [13]:
df = pd.read_excel("deepSeekR1Data.xlsx")

In [14]:
ref_col = 'Hop_5_Rationale_Target_Span (Ref)'
pred_col = 'Hop_5_Rationale_Target_Span'

In [15]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')
print(len(df))

872


In [16]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [17]:
tokenizer = WordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text)

In [18]:
def jaccard_similarity(a, b):
    A = set(tokenize(a))
    B = set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def max_pairwise_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    return max(
        jaccard_similarity(p, g)
        for p in pred_spans
        for g in gold_spans
    )

In [19]:
def max_pairwise_bertscore(
    pred_spans,
    gold_spans,
    model_type="HooshvareLab/bert-base-parsbert-uncased",
    device=None
):
    if not pred_spans or not gold_spans:
        return 0.0

    max_score = 0.0

    for p in pred_spans:
        P, R, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type=model_type,
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        max_score = max(max_score, F1.max().item())

    return max_score

In [20]:
def evaluate_rationales(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        print(index)
        max_pairwise_jaccard_val = max_pairwise_jaccard(row[pred_col], row[gold_col])
        jaccards.append(max_pairwise_jaccard_val)

        bertScore = max_pairwise_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "MaxPairwiseJaccard": sum(jaccards) / len(jaccards),
        "MaxPairwiseBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results = evaluate_rationales(df, gold_col=ref_col, pred_col=pred_col)

In [22]:
results

{'MaxPairwiseJaccard': 0.7288032694343052,
 'MaxPairwiseBERTScore': 0.853399859314118}

In [23]:
def jaccard(a, b):
    A, B = set(tokenize(a)), set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

In [24]:
def normal_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        scores.append(max(jaccard(p, g) for g in gold_spans))
    return sum(scores) / len(scores)

In [25]:
def normal_bertscore(pred_spans, gold_spans, device=None):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        _, _, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type="HooshvareLab/bert-base-parsbert-uncased",
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        scores.append(F1.max().item())
    return sum(scores) / len(scores)

In [26]:
def evaluate_rationales2(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        print(index)
        jaccard_val = normal_jaccard(row[pred_col], row[gold_col])
        jaccards.append(jaccard_val)

        bertScore = normal_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "normalJaccard": sum(jaccards) / len(jaccards),
        "normalBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results2 = evaluate_rationales2(df, gold_col=ref_col, pred_col=pred_col)

In [28]:
results2

{'normalJaccard': 0.6448933767658115, 'normalBERTScore': 0.8159464171630968}